# Machine Learning Lab Practical Exam — Complete Code Pack

**PCC353AID | Third Year AI & DS | Practical exam: 3 November 2026**

This notebook contains one self-contained program for each of the 11 experiments in the supplied manual. Run the setup cell once, then run the experiment you are revising. The datasets are generated locally or loaded from scikit-learn, so no CSV download is required.

**Requirements:** Python 3, NumPy, Pandas, Matplotlib, Scikit-learn.

In [ ]:
# Run once in Google Colab if any library is missing:
# %pip install numpy pandas matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

## Practical 1 — Feature Engineering
**Aim:** Handle missing numerical/categorical values, encode categories, and scale numerical features.

This small DataFrame intentionally contains missing values and categorical columns so every required operation is visible.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Step 1: Create/load a dataset
data = {
    "Age": [21, 22, np.nan, 24, 23],
    "Salary": [25000, 30000, 28000, np.nan, 35000],
    "Gender": ["Male", "Female", "Male", np.nan, "Female"],
    "City": ["Nashik", "Pune", "Nashik", "Mumbai", np.nan]
}
df = pd.DataFrame(data)

print("Original dataset:\n", df)
print("\nMissing values:\n", df.isnull().sum())

# Step 2: Handle missing numerical values
for col in ["Age", "Salary"]:
    df[col] = df[col].fillna(df[col].median())

# Step 3: Handle missing categorical values using mode
for col in ["Gender", "City"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# Step 4: One-hot encode categorical columns
encoded_df = pd.get_dummies(df, columns=["Gender", "City"], dtype=int)

# Step 5: Apply Min-Max scaling to numerical columns
scaler = MinMaxScaler()
encoded_df[["Age", "Salary"]] = scaler.fit_transform(encoded_df[["Age", "Salary"]])

print("\nCleaned and model-ready dataset:\n", encoded_df)
print("\nRemaining missing values:", encoded_df.isnull().sum().sum())

## Practical 2 — PCA and LDA
**Aim:** Reduce dimensions and compare classification accuracy using original, PCA-reduced, and LDA-reduced features.

Dataset: Wine dataset (13 numerical features, 3 classes). LDA can produce at most `number_of_classes - 1` components, so here we use 2.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Step 1: Load dataset
wine = load_wine()
X, y = wine.data, wine.target

# Step 2: Split before fitting preprocessing (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 3: Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 4: Train classifier on original features
model_original = LogisticRegression(max_iter=3000)
model_original.fit(X_train_scaled, y_train)
pred_original = model_original.predict(X_test_scaled)

# Step 5: PCA dimensionality reduction
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
model_pca = LogisticRegression(max_iter=3000)
model_pca.fit(X_train_pca, y_train)
pred_pca = model_pca.predict(X_test_pca)

# Step 6: LDA dimensionality reduction (uses class labels)
lda = LinearDiscriminantAnalysis(n_components=2)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)
model_lda = LogisticRegression(max_iter=3000)
model_lda.fit(X_train_lda, y_train)
pred_lda = model_lda.predict(X_test_lda)

# Step 7: Compare results
print("Original features:", X.shape[1], "| Accuracy:", round(accuracy_score(y_test, pred_original), 4))
print("PCA features:", X_train_pca.shape[1], "| Accuracy:", round(accuracy_score(y_test, pred_pca), 4))
print("LDA features:", X_train_lda.shape[1], "| Accuracy:", round(accuracy_score(y_test, pred_lda), 4))
print("PCA explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance retained by 2 PCs:", round(pca.explained_variance_ratio_.sum(), 4))

## Practical 3 — Multiple Linear Regression
**Aim:** Predict a continuous target using multiple independent variables and evaluate using MSE, RMSE, and R².

Dataset: scikit-learn Diabetes dataset.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Load dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

# Step 2: Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 3: Create and train model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 4: Predict
predictions = model.predict(X_test)

# Step 5: Evaluate
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print("Mean Squared Error (MSE):", round(mse, 3))
print("Root Mean Squared Error (RMSE):", round(rmse, 3))
print("R2 Score:", round(r2, 3))
print("Regression coefficients:", model.coef_)

## Practical 4 — Polynomial Regression
**Aim:** Train polynomial models of different degrees and observe underfitting/overfitting through training and testing error.

A reproducible curved dataset is generated locally. Degree 1 may underfit; a very high degree may overfit.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Generate nonlinear data
rng = np.random.RandomState(42)
X = np.linspace(-3, 3, 100).reshape(-1, 1)
y = 0.5 * X[:, 0]**2 + X[:, 0] + rng.normal(0, 1.0, size=100)

# Step 2: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 3: Train models with different polynomial degrees
for degree in [1, 2, 3, 5, 10]:
    model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)

    print(f"Degree {degree}: Train MSE={train_mse:.3f}, Test MSE={test_mse:.3f}, Test R2={test_r2:.3f}")

# Step 4: Plot the data and one example model (degree 3)
model3 = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
model3.fit(X_train, y_train)
order = np.argsort(X[:, 0])
plt.scatter(X[:, 0], y, label="Data")
plt.plot(X[order, 0], model3.predict(X[order]), label="Degree 3 model")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Polynomial Regression")
plt.legend()
plt.show()

## Practical 5 — Ridge and Lasso Regression
**Aim:** Compare L2 (Ridge) and L1 (Lasso) regularization and observe their effect on coefficients and prediction performance.

Dataset: Diabetes dataset. Features are standardized because regularization is scale-sensitive.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Load and split data
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 2: Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 3: Train Ridge and Lasso models
models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.5, max_iter=10000)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    print(f"\n{name} Regression")
    print("MSE:", round(mean_squared_error(y_test, pred), 3))
    print("R2:", round(r2_score(y_test, pred), 3))
    print("Coefficients:", np.round(model.coef_, 3))
    print("Number of zero coefficients:", int(np.sum(model.coef_ == 0)))

## Practical 6 — Logistic Regression
**Aim:** Perform binary classification and evaluate with confusion matrix, precision, recall, and F1-score.

Dataset: Breast Cancer Wisconsin dataset. Here, malignant is mapped to positive class `1` for this demonstration.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

# Step 1: Load dataset
cancer = load_breast_cancer()
X = cancer.data
# Original target: 0 = malignant, 1 = benign. Make malignant the positive class.
y = (cancer.target == 0).astype(int)

# Step 2: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 3: Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 4: Train Logistic Regression
model = LogisticRegression(max_iter=3000)
model.fit(X_train_scaled, y_train)

# Step 5: Predict and evaluate
pred = model.predict(X_test_scaled)
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, pred, zero_division=0), 4))
print("\nClassification Report:\n", classification_report(y_test, pred, zero_division=0))

## Practical 7 — Support Vector Machine (SVM)
**Aim:** Compare Linear and RBF (nonlinear) SVM kernels using classification metrics.

Dataset: Breast Cancer Wisconsin dataset. Features are standardized before SVM because SVM is scale-sensitive.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Step 1: Load dataset
cancer = load_breast_cancer()
X = cancer.data
y = (cancer.target == 0).astype(int)  # 1 = malignant, 0 = benign

# Step 2: Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 3: Compare linear and RBF kernels
for kernel in ["linear", "rbf"]:
    model = SVC(kernel=kernel, C=1.0, gamma="scale")
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)

    print(f"\nSVM with {kernel} kernel")
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Precision:", round(precision_score(y_test, pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, pred, zero_division=0), 4))
    print("F1-score:", round(f1_score(y_test, pred, zero_division=0), 4))

### Optional SVM dataset variant from your manual: Pima Indians Diabetes
Your manual (pages 34–36) describes the Pima dataset with 768 records, 8 numerical features, and `Outcome` as the target. The main program above uses the built-in Breast Cancer dataset so it runs without a separate CSV. When your teacher specifically asks for Pima, place `diabetes.csv` in the working folder (or upload it in Colab) and use this version:


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# CSV must contain the 8 input columns and the target column named Outcome
df = pd.read_csv("diabetes.csv")
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for kernel in ["linear", "rbf"]:
    model = SVC(kernel=kernel, C=1.0, gamma="scale")
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    print(f"\n{kernel} kernel")
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Precision:", round(precision_score(y_test, pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, pred, zero_division=0), 4))
    print("F1-score:", round(f1_score(y_test, pred, zero_division=0), 4))


## Practical 8 — K-Means Clustering with Elbow and Silhouette Methods
**Aim:** Find a suitable number of clusters using WCSS (Elbow) and Silhouette score.

Dataset: Iris dataset. Clustering is unsupervised; the species labels are not used for training.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Step 1: Load and scale feature data
iris = load_iris()
X = iris.data
X_scaled = StandardScaler().fit_transform(X)

# Step 2: Elbow method (WCSS / inertia)
wcss = []
k_values = range(1, 11)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.plot(list(k_values), wcss, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("WCSS / Inertia")
plt.title("Elbow Method")
plt.show()

# Step 3: Silhouette method (K must be at least 2)
silhouette_scores = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_scores[k] = silhouette_score(X_scaled, labels)

print("Silhouette scores:")
for k, score in silhouette_scores.items():
    print(f"K={k}: {score:.4f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print("Best K according to Silhouette Method:", best_k)

# Step 4: Final clustering
final_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_model.fit_predict(X_scaled)
print("Cluster labels:", cluster_labels)
print("Cluster sizes:", np.bincount(cluster_labels))

## Practical 9 — DBSCAN Clustering
**Aim:** Identify dense clusters and noise/outlier points using DBSCAN.

Dataset: Iris dataset. Feature scaling is important because DBSCAN uses distance. `eps` and `min_samples` may be tuned depending on the dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

# Step 1: Load and scale dataset
iris = load_iris()
X = iris.data
X_scaled = StandardScaler().fit_transform(X)

# Step 2: Apply DBSCAN
model = DBSCAN(eps=0.8, min_samples=5)
labels = model.fit_predict(X_scaled)

# Step 3: Count clusters and noise
unique_labels = set(labels)
number_of_clusters = len(unique_labels - {-1})
noise_count = int(np.sum(labels == -1))
print("Cluster labels:", sorted(unique_labels))
print("Number of clusters (excluding noise):", number_of_clusters)
print("Number of noise points:", noise_count)

# Step 4: Visualize in two dimensions using PCA
X_2d = PCA(n_components=2).fit_transform(X_scaled)
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("DBSCAN Clusters (-1 represents noise)")
plt.show()

## Practical 10 — Ensemble Learning: Random Forest vs Baseline
**Aim:** Compare a baseline Decision Tree with Random Forest using classification metrics.

Dataset: Breast Cancer Wisconsin dataset. Random Forest combines many decision trees and uses majority voting for classification.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Step 1: Load and split dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Define baseline and ensemble models
models = {
    "Decision Tree (baseline)": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

# Step 3: Train, predict, compare
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"\n{name}")
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Precision:", round(precision_score(y_test, pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, pred, zero_division=0), 4))
    print("F1-score:", round(f1_score(y_test, pred, zero_division=0), 4))

## Practical 11 — Q-Learning using Grid World
**Aim:** Train an agent to reach a goal using rewards, Q-values, and an epsilon-greedy exploration strategy.

This is a small self-contained 4 × 4 Grid World. No Gym package is needed. The agent starts at the top-left and tries to reach the bottom-right while avoiding blocked cells.

In [ ]:
import numpy as np
import random

# Step 1: Define the Grid World
grid_size = 4
start = (0, 0)
goal = (3, 3)
obstacles = {(1, 1), (1, 2)}
actions = [0, 1, 2, 3]  # 0=Up, 1=Down, 2=Left, 3=Right
action_symbols = {0: "U", 1: "D", 2: "L", 3: "R"}

# Convert (row, col) to one Q-table state index
def state_id(position):
    return position[0] * grid_size + position[1]

# Environment transition function
def step(position, action):
    row, col = position
    if action == 0:       # Up
        new_position = (max(0, row - 1), col)
    elif action == 1:     # Down
        new_position = (min(grid_size - 1, row + 1), col)
    elif action == 2:     # Left
        new_position = (row, max(0, col - 1))
    else:                 # Right
        new_position = (row, min(grid_size - 1, col + 1))

    if new_position in obstacles:
        return position, -5, False
    if new_position == goal:
        return new_position, 20, True
    if new_position == position:
        return new_position, -2, False
    return new_position, -1, False

# Step 2: Initialize Q-table and learning parameters
q_table = np.zeros((grid_size * grid_size, len(actions)))
alpha = 0.1       # Learning rate
gamma = 0.9       # Discount factor
epsilon = 1.0     # Initial exploration probability
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 3000

# Step 3: Train the agent
for episode in range(episodes):
    position = start
    for step_number in range(50):
        s = state_id(position)

        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = random.choice(actions)  # Exploration
        else:
            action = int(np.argmax(q_table[s]))  # Exploitation

        next_position, reward, done = step(position, action)
        next_s = state_id(next_position)

        # Q-learning update
        if done:
            target = reward
        else:
            target = reward + gamma * np.max(q_table[next_s])
        q_table[s, action] += alpha * (target - q_table[s, action])

        position = next_position
        if done:
            break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

# Step 4: Display learned policy
print("Learned policy (S=start, G=goal, #=obstacle):")
for row in range(grid_size):
    row_symbols = []
    for col in range(grid_size):
        pos = (row, col)
        if pos == start:
            symbol = "S"
        elif pos == goal:
            symbol = "G"
        elif pos in obstacles:
            symbol = "#"
        else:
            best_action = int(np.argmax(q_table[state_id(pos)]))
            symbol = action_symbols[best_action]
        row_symbols.append(symbol)
    print(" ".join(row_symbols))

# Step 5: Test greedy policy after learning
position = start
path = [position]
for _ in range(20):
    action = int(np.argmax(q_table[state_id(position)]))
    next_position, reward, done = step(position, action)
    if next_position == position and not done:
        break
    position = next_position
    path.append(position)
    if done:
        break

print("\nGreedy path after training:", path)
print("Reached goal:", position == goal)

---
## Last-minute practical checklist

- Know the order: import → load/create data → separate X and y (when supervised) → train-test split → preprocess → model fit → predict → evaluate.
- Fit scalers/encoders on training data only, then use `transform` on test data.
- Regression metrics: MSE, RMSE, R².
- Classification metrics: confusion matrix, accuracy, precision, recall, F1-score.
- Clustering: no target labels are needed for K-Means or DBSCAN.
- Reproducibility: `random_state=42` helps get repeatable results.
- For Q-Learning, be able to explain state, action, reward, Q-table, alpha, gamma, epsilon, exploration, and exploitation.

**Important:** Exact metric values can vary with library versions or changed parameters. Explain what each metric means rather than memorizing a number.